# MNIST Classification with Fully Connected Layers in PyTorch

This notebook walks through building a simple feedforward neural network to classify handwritten digits from the MNIST dataset using PyTorch.


## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Load and Explore the Data

We use `torchvision` to download MNIST and apply a basic normalization transform.
The images are 28×28 grayscale, giving 784 input features per sample.

In [ ]:
# Transform: flatten to 784-dim vector and normalize to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")
print(f"Input shape      : {train_dataset[0][0].shape}")


In [ ]:
# Visualize a few training samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i*100 + 25]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.suptitle("Sample MNIST Images", fontsize=14)
plt.tight_layout()
plt.show()


## 3. Define the Model

We build a simple 3-layer fully connected network:

```
Input (784) → FC(512) → ReLU → Dropout → FC(256) → ReLU → Dropout → FC(10)
```

- **Dropout** (p=0.3) helps regularize and prevent overfitting.
- The final layer outputs raw logits for 10 classes (digits 0–9).
- We use `CrossEntropyLoss`, which internally applies softmax.

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self, input_size=784, hidden1=256, hidden2=64, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                          # (B, 1, 28, 28) -> (B, 784)
            nn.Linear(input_size, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, num_classes)        # logits
        )

    def forward(self, x):
        return self.net(x)

model = MNISTClassifier().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")


## 4. Train the Model

We use:
- **Cross-entropy loss** (suitable for multi-class classification)
- **Adam optimizer** with a learning rate of 1e-3
- A simple learning-rate scheduler that decays LR by 0.5 every 5 epochs

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


NUM_EPOCHS = 15
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, test_loader, criterion)
    # scheduler.step()
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}% | "
          f"Val Loss: {va_loss:.4f}  Acc: {va_acc*100:.2f}%")


In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], label="Train")
ax1.plot(epochs, history["val_loss"],   label="Validation")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves"); ax1.legend()

ax2.plot(epochs, [a*100 for a in history["train_acc"]], label="Train")
ax2.plot(epochs, [a*100 for a in history["val_acc"]],   label="Validation")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy Curves"); ax2.legend()

plt.tight_layout()
plt.show()


## 5. Evaluate the Model

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=[str(i) for i in range(10)]))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 6. Visualize Predictions

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    logits = model(images.to(device))
    probs  = torch.softmax(logits, dim=1).cpu()
    preds  = probs.argmax(1)

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = images[i].squeeze()
    pred, true = preds[i].item(), labels[i].item()
    conf = probs[i, pred].item()
    ax.imshow(img, cmap="gray")
    color = "green" if pred == true else "red"
    ax.set_title(f"Pred: {pred} ({conf*100:.0f}%)\nTrue: {true}", color=color, fontsize=8)
    ax.axis("off")
plt.suptitle("Predictions (green = correct, red = wrong)", fontsize=13)
plt.tight_layout()
plt.show()
